## Feature Importance

### General Libraries

In [ ]:
import pandas as pd
import warnings
import seaborn as sns

warnings.filterwarnings("ignore")

### 1. Data Exploration

In [ ]:
# read the data
df = pd.read_csv("D:\ProjectData\water_potability.csv")

# check data frame shape
df.shape

In [ ]:
# check column names and first few rows
df.head()

In [ ]:
# check data types
df.dtypes

In [ ]:
# check for missing values
df.isnull().sum()

In [ ]:
# check distribution of target variable
sns.countplot(data=df, x="Potability")

### 2. Data Preprocessing

In [ ]:
# remove rows with na
df.dropna(inplace = True)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn import preprocessing

# split data into input and target value
X = df.drop(["Potability"], 1)
y = df.Potability

# standardization of input data
scaler = preprocessing.StandardScaler()

X_normalized_values = scaler.fit_transform(X.values)

X_normalized = pd.DataFrame(X_normalized_values, columns=X.columns, index=X.index)


# split into training and test data
X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size=0.10, stratify = y, random_state=42)


### 3. Feature Importance

#### Logistic Regression - Coefficients

In [ ]:
from sklearn.linear_model import LogisticRegression

# initialize logistic regression model and fit to training data
logreg = LogisticRegression()
logreg.fit(X_train,y_train)

# obtain coefficients
coefficients = logreg.coef_.reshape(-1)

# present features and absolute values of coefficients in a dataframe
logreg_fi_df = pd.DataFrame(data = {'Feature': X.columns, 'Importance': np.abs(coefficients)}).sort_values(by = "Importance", ascending = False)
logreg_fi_df

#### Gradient Boosted Trees 

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# initialize gradient boosted trees and fit to training data
clf = GradientBoostingClassifier(random_state=42)
clf.fit(X_train, y_train)

# calculate feature importance
feature_importance = clf.feature_importances_

# present features and standardized decrease in gini impurity in a dataframe
gbt_fi_df = pd.DataFrame({'Feature': X.columns, 'Importance': feature_importance})
gbt_fi_df = gbt_fi_df.sort_values(by='Importance', ascending=False)
gbt_fi_df

#### Neural Network - Feature permutation

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score

# initialise neural network and fit to training data
mlp = MLPClassifier(random_state=42, hidden_layer_sizes = (120,))
mlp.fit(X_train, y_train)

# calculate permutation importance
perm_importance = permutation_importance(mlp, X_test, y_test, n_repeats=5, random_state=42)

# present features and feature permutation importance in a dataframe
feature_importances_df = pd.DataFrame({
    'feature': X.columns,
    'importance': perm_importance.importances_mean
})
feature_importances_df = feature_importances_df.sort_values(by='importance', ascending=False)
feature_importances_df

#### Neural Network (SHAP)

In [ ]:
import shap

# compute SHAP values
explainer = shap.KernelExplainer(mlp.predict_proba, shap.sample(X_train, 100))
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, plot_type="bar")